# Feature Engineering

This notebook builds the first incremental feature sets for the attendance-prediction task.

## 1. Load Prediction Instances

The input to this notebook is the prediction-instance dataset created in Notebook 02. Each row represents one class observed at one prediction horizon together with the booking snapshot available at prediction time.

In [ ]:
from pathlib import Path

import pandas as pd

from src.data import load_data, prepare_attendance, prepare_booking_events
from src.feature_engineering import (
    add_baseline_features,
    add_booking_dynamics_features,
    add_class_context_features,
    add_historical_features,
    add_member_history_features,
    add_member_reliability_features,
)

USE_SYNTHETIC = False

DATA_DIR = Path("..") / "data" / "processed"
PREDICTION_INSTANCES_PATH = DATA_DIR / "prediction_instances.parquet"
BASELINE_FEATURES_PATH = DATA_DIR / "baseline_features.parquet"
BASELINE_PLUS_HISTORICAL_FEATURES_PATH = (
    DATA_DIR / "baseline_plus_historical_features.parquet"
)
BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_FEATURES_PATH = (
    DATA_DIR / "baseline_plus_historical_plus_member_features.parquet"
)
BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_PLUS_RELIABILITY_FEATURES_PATH = (
    DATA_DIR
    / "baseline_plus_historical_plus_member_plus_reliability_features.parquet"
)
BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_PLUS_RELIABILITY_PLUS_BOOKING_DYNAMICS_FEATURES_PATH = (
    DATA_DIR
    / "baseline_plus_historical_plus_member_plus_reliability_plus_booking_dynamics_features.parquet"
)
BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_PLUS_RELIABILITY_PLUS_DYNAMICS_PLUS_CONTEXT_FEATURES_PATH = (
    DATA_DIR
    / "baseline_plus_historical_plus_member_plus_reliability_plus_dynamics_plus_context_features.parquet"
)

prediction_identifier_columns = [
    "studio",
    "course",
    "class_start",
    "prediction_horizon",
]

preview_identifier_columns = (
    prediction_identifier_columns + ["prediction_time", "event_timestamp"]
)

class_columns = ["studio", "course", "class_start"]

baseline_feature_columns = [
    "attendance_count",
    "available_spots",
    "occupancy_rate",
    "is_full",
    "has_waiting_list",
    "weekday",
    "class_hour",
]

historical_feature_columns = [
    f"{group}_attendance_{stat}_{window}"
    for group in ["course", "instructor", "studio"]
    for window in ["30d", "90d", "365d"]
    for stat in ["mean", "std", "count"]
]

member_history_feature_columns = [
    f"{feature}_{window}"
    for feature in [
        "member_attendance_count_mean",
        "members_with_history_count",
        "members_with_history_share",
        "member_course_affinity_mean",
        "member_instructor_affinity_mean",
    ]
    for window in ["30d", "90d", "365d"]
]

member_reliability_feature_columns = [
    f"{feature}_{window}"
    for feature in [
        "member_booked_at_horizon_count_mean",
        "members_with_reliability_history_count",
        "members_with_reliability_history_share",
        "member_show_up_rate_mean",
    ]
    for window in ["30d", "90d", "365d"]
]

booking_dynamics_feature_columns = [
    f"{feature}_{window}"
    for feature in [
        "observed_booking_increase_events",
        "observed_booking_decrease_events",
        "observed_net_booking_change",
    ]
    for window in ["6h", "24h", "72h"]
] + [
    "max_observed_attendance_before_prediction",
    "attendance_drop_from_observed_peak",
]

class_context_feature_columns = [
    "total_demand_count",
    "total_demand_fill_ratio",
    "course_instructor_attendance_mean_90d",
    "course_instructor_history_count_90d",
]


In [ ]:
prediction_instances = pd.read_parquet(PREDICTION_INSTANCES_PATH)
prediction_instances.shape

In [ ]:
prediction_instances.head(3)

## 2. Baseline Features

The baseline feature group uses only information that is directly available from the current class and the current booking snapshot.

In [ ]:
features = add_baseline_features(prediction_instances)
features.shape

In [ ]:
features[
    preview_identifier_columns + baseline_feature_columns + ["final_attendance_count"]
].head(3)

### Validate and Save

The baseline step must preserve the prediction instances while adding the new snapshot-based features. The incremental baseline state is saved before historical features are added.

In [ ]:
assert len(features) == len(prediction_instances)
assert features["final_attendance_count"].equals(
    prediction_instances["final_attendance_count"]
)
assert prediction_instances["attendance_list"].map(len).equals(
    features["attendance_count"]
)
assert prediction_instances["waiting_list_length"].equals(
    features["waiting_list_length"]
)
assert "waiting_count" not in features.columns
assert features.duplicated(prediction_identifier_columns).sum() == 0

In [ ]:
features.to_parquet(BASELINE_FEATURES_PATH, index=False)
BASELINE_FEATURES_PATH

## 3. Historical Features

Historical features summarize attendance patterns from classes that had already occurred by the prediction time. They use the full historical attendance dataset while ensuring that no future attendance outcomes are included.

In [ ]:
_, attendance_raw = load_data(use_synthetic=USE_SYNTHETIC)
attendance_history = prepare_attendance(attendance_raw)

attendance_history = attendance_history.dropna()
attendance_history = attendance_history.loc[
    ~attendance_history.duplicated(
        subset=class_columns,
        keep=False,
    )
]

features = add_historical_features(features, attendance_history)
features.shape

In [ ]:
historical_preview_columns = preview_identifier_columns + [
    "course_attendance_mean_30d",
    "course_attendance_count_30d",
    "course_attendance_mean_365d",
    "course_attendance_count_365d",
    "instructor_attendance_mean_90d",
    "instructor_attendance_count_90d",
    "studio_attendance_mean_365d",
    "studio_attendance_count_365d",
]

features[historical_preview_columns].head(3)

### Validate and Save

The historical step must preserve the existing prediction instances and target while ensuring that only classes before each prediction time contribute to the historical summaries.

In [ ]:
assert len(features) == len(prediction_instances)
assert features["final_attendance_count"].equals(
    prediction_instances["final_attendance_count"]
)
assert features.duplicated(prediction_identifier_columns).sum() == 0

historical_count_columns = [
    column for column in features.columns
    if "_attendance_count_" in column
]

assert (features[historical_count_columns] >= 0).all().all()

In [ ]:
features.to_parquet(
    BASELINE_PLUS_HISTORICAL_FEATURES_PATH,
    index=False,
)
BASELINE_PLUS_HISTORICAL_FEATURES_PATH

## 4. Member-Level Features

These features summarize the historical attendance behavior and affinities of members who are currently booked at prediction time.

In [ ]:
features = add_member_history_features(features, attendance_history)
features.shape

In [ ]:
member_history_preview_columns = preview_identifier_columns + [
    "member_attendance_count_mean_30d",
    "members_with_history_count_30d",
    "members_with_history_share_30d",
    "member_course_affinity_mean_90d",
    "member_instructor_affinity_mean_90d",
    "member_attendance_count_mean_365d",
    "members_with_history_count_365d",
]

features[member_history_preview_columns].head(3)

### Validate and Save

The member-level step must preserve the existing prediction instances while producing interpretable member-history summaries with bounded shares and affinities.

In [ ]:
member_history_count_columns = [
    column for column in member_history_feature_columns
    if column.startswith("members_with_history_count")
]
member_history_share_columns = [
    column for column in member_history_feature_columns
    if column.startswith("members_with_history_share")
]
member_affinity_columns = [
    column for column in member_history_feature_columns
    if "affinity" in column
]

assert len(features) == len(prediction_instances)
assert features["final_attendance_count"].equals(
    prediction_instances["final_attendance_count"]
)
assert features.duplicated(prediction_identifier_columns).sum() == 0
assert (features[member_history_count_columns] >= 0).all().all()
assert features[member_history_count_columns].le(
    features["attendance_count"],
    axis=0,
).all().all()

for column in member_history_share_columns:
    non_missing = features[column].dropna()
    assert non_missing.between(0, 1).all()

for column in member_affinity_columns:
    non_missing = features[column].dropna()
    assert non_missing.between(0, 1).all()

In [ ]:
features.to_parquet(
    BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_FEATURES_PATH,
    index=False,
)
BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_FEATURES_PATH

## 5. Member Reliability Features

These features estimate how reliably the members currently booked for a class have historically attended when they were already booked at the same prediction horizon before past classes.

In [ ]:
booking_events_raw, _ = load_data(use_synthetic=USE_SYNTHETIC)
booking_events_history = prepare_booking_events(booking_events_raw)
booking_events_history["event_order"] = range(len(booking_events_history))

features = add_member_reliability_features(
    features,
    booking_events_history,
    attendance_history,
)
features.shape

In [ ]:
member_reliability_preview_columns = preview_identifier_columns + [
    "member_booked_at_horizon_count_mean_30d",
    "members_with_reliability_history_count_30d",
    "members_with_reliability_history_share_30d",
    "member_show_up_rate_mean_30d",
    "member_booked_at_horizon_count_mean_365d",
    "members_with_reliability_history_count_365d",
    "member_show_up_rate_mean_365d",
]

features[member_reliability_preview_columns].head(3)

### Validate and Save

The reliability step must preserve the existing prediction instances while producing bounded show-up summaries from same-horizon historical booking states.

In [ ]:
reliability_count_columns = [
    column for column in member_reliability_feature_columns
    if column.startswith("members_with_reliability_history_count")
]
reliability_share_columns = [
    column for column in member_reliability_feature_columns
    if column.startswith("members_with_reliability_history_share")
]
show_up_rate_columns = [
    column for column in member_reliability_feature_columns
    if column.startswith("member_show_up_rate_mean")
]

assert len(features) == len(prediction_instances)
assert features["final_attendance_count"].equals(
    prediction_instances["final_attendance_count"]
)
assert features.duplicated(prediction_identifier_columns).sum() == 0
assert (features[reliability_count_columns] >= 0).all().all()
assert features[reliability_count_columns].le(
    features["attendance_count"],
    axis=0,
).all().all()

for column in reliability_share_columns:
    non_missing = features[column].dropna()
    assert non_missing.between(0, 1).all()

for column in show_up_rate_columns:
    non_missing = features[column].dropna()
    assert non_missing.between(0, 1).all()

In [ ]:
features.to_parquet(
    BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_PLUS_RELIABILITY_FEATURES_PATH,
    index=False,
)
BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_PLUS_RELIABILITY_FEATURES_PATH

## 6. Booking Dynamics Features

These features summarize how the observed booking state of the current class changed before prediction time. They use only booking snapshots available at or before the current prediction time and count observed increases and decreases in booked attendance across short recency windows.

In [ ]:
features = add_booking_dynamics_features(
    features,
    booking_events_history,
)
features.shape

In [ ]:
booking_dynamics_preview_columns = preview_identifier_columns + [
    "observed_booking_increase_events_6h",
    "observed_booking_decrease_events_6h",
    "observed_net_booking_change_6h",
    "observed_booking_increase_events_24h",
    "observed_booking_decrease_events_24h",
    "observed_net_booking_change_24h",
    "max_observed_attendance_before_prediction",
    "attendance_drop_from_observed_peak",
]

features[booking_dynamics_preview_columns].head(3)

### Validate and Save

The booking-dynamics step must preserve the prediction instances while summarizing only the booking-state transitions and attendance peaks observable by prediction time.

In [ ]:
booking_dynamics_count_columns = [
    column for column in booking_dynamics_feature_columns
    if "increase_events" in column or "decrease_events" in column
]

assert len(features) == len(prediction_instances)
assert features["final_attendance_count"].equals(
    prediction_instances["final_attendance_count"]
)
assert features.duplicated(prediction_identifier_columns).sum() == 0
assert (features[booking_dynamics_count_columns] >= 0).all().all()
assert features["max_observed_attendance_before_prediction"].ge(
    features["attendance_count"]
).all()
assert (features["attendance_drop_from_observed_peak"] >= 0).all()
assert (
    features["max_observed_attendance_before_prediction"]
    - features["attendance_count"]
).eq(features["attendance_drop_from_observed_peak"]).all()

In [ ]:
features.to_parquet(
    BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_PLUS_RELIABILITY_PLUS_BOOKING_DYNAMICS_FEATURES_PATH,
    index=False,
)
BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_PLUS_RELIABILITY_PLUS_BOOKING_DYNAMICS_FEATURES_PATH

## 7. Class Context Features

These features describe total demand relative to capacity at prediction time and attendance during the previous 90 days for the same course-instructor combination. Existing baseline features remain unchanged.

In [ ]:
features = add_class_context_features(
    features,
    attendance_history,
)
features.shape

In [ ]:
features[class_context_feature_columns].head(3)

### Validate and Save

The class-context step must preserve the prediction instances while using only the current snapshot and historical classes that started before prediction time.

In [ ]:
assert len(features) == len(prediction_instances)
assert features["final_attendance_count"].equals(
    prediction_instances["final_attendance_count"]
)
assert features.duplicated(prediction_identifier_columns).sum() == 0
assert features["total_demand_count"].ge(features["attendance_count"]).all()
assert features["total_demand_count"].eq(
    features["attendance_count"] + features["waiting_list_length"]
).all()
assert (features["total_demand_fill_ratio"].dropna() >= 0).all()
assert (features["course_instructor_history_count_90d"] >= 0).all()

In [ ]:
features.to_parquet(
    BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_PLUS_RELIABILITY_PLUS_DYNAMICS_PLUS_CONTEXT_FEATURES_PATH,
    index=False,
)
BASELINE_PLUS_HISTORICAL_PLUS_MEMBER_PLUS_RELIABILITY_PLUS_DYNAMICS_PLUS_CONTEXT_FEATURES_PATH